# Study 938 — Open or Close 🔔

**The same timing rule, filled at tomorrow's open or at tomorrow's close. Does it matter?**

Every tactical back-test quietly picks a fill venue and almost none report the other choice.
We take Faber's moving-average filter — hold the ETF while its period-end close is above the
mean of the last `L` period-end closes, else hold **BIL** (T-bills) — run it in two flavours
(**10-month**, month-end; **20-week**, week-end) on **SPY / IWM / EEM / EFA**, and fill it
twice: once at the **open** of the next session, once at its **close**.

Window 2007-05-30 → 2026-06-30 (4,802 days), 2 bps one-way × NAV,
long-only so no borrow. Adjusted OHLC (`auto_adjust=True`): the intraday leg is **price-only**,
the overnight leg carries the dividend.

*Real numbers below are the frozen headline (`docs/results.md`, Fingerprint `f36d90ae4fdc`,
as-of 2026-06-30). The only live cells run the offline **synthetic** control and say so.*


## 1. Why the choice looks free — and why it isn't obviously free

Fix the rule and fix the lag: the signal is known at tonight's close, and you fill tomorrow. On every day you *don't* trade, the two versions of you own exactly the same thing. The only days they differ are the handful when the rule flips — and on those days the difference is one single session's move, the **open-to-close** move.

So the whole question shrinks to: *is the trading session after a buy signal systematically greener than the session after a sell signal?* If trends carry over into the next day, filling at the open should win. If the opening auction is just the expensive place to trade, filling at the close should win.

## 2. The first answer looks exciting

In [1]:
R = {'m_tapes': {'SPY': (31, -4.8, -0.16, -2.9, 0.45, 0.587, 0.589), 'IWM': (37, 50.7, 1.04, 25.1, 0.59, 0.363, 0.331), 'EEM': (37, 66.6, 2.4, 32.9, 0.7, 0.313, 0.27), 'EFA': (33, 35.4, 1.69, 19.6, 0.61, 0.299, 0.272)}, 'm_gap': 37.0, 'm_t': 1.53}
print('10-month rule — gap = (fill at the open) minus (fill at the close)')
for tk, v in R['m_tapes'].items():
    print(f"  {tk}: {v[0]:3d} trades   {v[1]:+7.1f} bps/yr   HAC t {v[2]:+5.2f}   "
          f"open wins {v[4]:.0%} of fills")
print(f"  POOLED: {R['m_gap']:+.1f} bps/yr (HAC t {R['m_t']:+.2f})")

10-month rule — gap = (fill at the open) minus (fill at the close)
  SPY:  31 trades      -4.8 bps/yr   HAC t -0.16   open wins 45% of fills
  IWM:  37 trades     +50.7 bps/yr   HAC t +1.04   open wins 59% of fills
  EEM:  37 trades     +66.6 bps/yr   HAC t +2.40   open wins 70% of fills
  EFA:  33 trades     +35.4 bps/yr   HAC t +1.69   open wins 61% of fills
  POOLED: +37.0 bps/yr (HAC t +1.53)


Three of the four tapes lean the same way and the pooled book gains **+37.0 bps/yr** from filling at the open. Emerging markets even clears the significance bar on its own (*t* = +2.40). This is the point at which a lot of back-tests would stop, declare an execution edge, and hard-code "fill at the open" into the production script.

## 3. Then you run a faster version of the same filter

Same four tapes, same window, same costs — only now the filter is rebalanced weekly instead of monthly, which roughly triples the number of fills and therefore the statistical power. Note it is a *shorter* rule too (20 weeks ≈ 100 sessions of lookback against ≈ 210 for ten months), so this is a cousin, not a clone.

In [2]:
W = {'w_tapes': {'SPY': (88, -35.8, -0.81, -7.6, 0.5, 0.588, 0.615), 'IWM': (114, -19.0, -0.29, -3.1, 0.46, 0.382, 0.392), 'EEM': (102, -26.2, -0.54, -4.8, 0.45, 0.333, 0.349), 'EFA': (96, -23.4, -0.66, -4.6, 0.45, 0.412, 0.432)}, 'w_gap': -26.1, 'w_t': -0.8}
print('20-week rule — same tapes, ~3x the fills')
for tk, v in W['w_tapes'].items():
    print(f"  {tk}: {v[0]:3d} trades   {v[1]:+7.1f} bps/yr   HAC t {v[2]:+5.2f}   "
          f"open wins {v[4]:.0%} of fills")
print(f"  POOLED: {W['w_gap']:+.1f} bps/yr (HAC t {W['w_t']:+.2f})  <- the sign flipped")

20-week rule — same tapes, ~3x the fills
  SPY:  88 trades     -35.8 bps/yr   HAC t -0.81   open wins 50% of fills
  IWM: 114 trades     -19.0 bps/yr   HAC t -0.29   open wins 46% of fills
  EEM: 102 trades     -26.2 bps/yr   HAC t -0.54   open wins 45% of fills
  EFA:  96 trades     -23.4 bps/yr   HAC t -0.66   open wins 45% of fills
  POOLED: -26.1 bps/yr (HAC t -0.80)  <- the sign flipped


**All four flip.** The same idea, measured with more data, says the opposite: filling at the open now *loses* 26 bps/yr. When a result reverses the moment you give it more observations, the honest reading is that it was never there.

> 🔬 **For the quants** — the two rules are not independent samples of the same quantity, but they share the tapes, the window and the costs, so a genuine venue effect would have to show up in both. Bootstrap CIs on the pooled gap: monthly [-12, +85] bps/yr, weekly [-91, +36] — both straddle zero.

## 4. Every fill on the desk, in one line

In [3]:
P = {'n_trades': 538, 'per_trade_bps': 1.39, 'per_trade_t': 0.33, 'wins': 267, 'win_rate': 49.6, 'wilson': (45.4, 53.8), 'n_dates': 361, 'per_trade_t_clu': 0.23, 'clu_win': (44.1, 55.1)}
print(f"all {P['n_trades']} fills, both rules, all four tapes")
print(f"  mean slippage of the open fill : {P['per_trade_bps']:+.2f} bps  (t = {P['per_trade_t']:+.2f})")
print(f"  the open fill won              : {P['wins']}/{P['n_trades']} = {P['win_rate']:.1f}%")
print(f"  95% confidence on that win rate: [{P['wilson'][0]:.1f}%, {P['wilson'][1]:.1f}%]")
print(f"\n  but the fills are not independent - they sit on only {P['n_dates']} dates")
print(f"  clustered on the trade date    : t = {P['per_trade_t_clu']:+.2f}, "
      f"win rate 95% CI [{P['clu_win'][0]:.1f}%, {P['clu_win'][1]:.1f}%]")
print('\n  -> a coin flip, and slightly more of one once you correct for that.')

all 538 fills, both rules, all four tapes
  mean slippage of the open fill : +1.39 bps  (t = +0.33)
  the open fill won              : 267/538 = 49.6%
  95% confidence on that win rate: [45.4%, 53.8%]

  but the fills are not independent - they sit on only 361 dates
  clustered on the trade date    : t = +0.23, win rate 95% CI [44.1%, 55.1%]

  -> a coin flip, and slightly more of one once you correct for that.


## 5. The coin flip is free — but the noise it buys is not

Zero expected return does not mean zero consequence. Across the eight rule × tape cells the *realised* venue gap ranges from **-35.8** to **+66.6 bps/yr** (sd 40). So an arbitrary, unexamined line in your execution code moves the reported CAGR by roughly **±0.4 pp/yr** — enough to make a mediocre strategy look respectable, or the reverse. It is the same disease as rebalance-timing luck ([Study 836](../../836-timing-luck/)), one dimension down: not *which day* you trade, but *what time of day*.

## 6. Is the harness even awake? (offline synthetic — not the real tape)

A null result is only worth reading if the test could have found something. The synthetic tape below books a *fixed* daily move disproportionately into the trading session after a strong or weak month — a planted venue edge. The close-executed arm is mathematically untouched by the planting, so only the open arm can move.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from open_close_exec import data, strategy as st
for ss, tag in [(1.0, 'planted venue edge'), (0.0, 'null (no edge)')]:
    t = np.array([st.synthetic_detect(
            data.synthetic_daily(signal_strength=ss, seed=938 + 11 * s)[0])['gap_t_hac']
        for s in range(8)])
    print(f"SYNTHETIC {tag:20s}: mean HAC t {t.mean():+.2f}, "
          f"fires |t|>=2 on {int((abs(t) >= 2).sum())}/8 seeds")

SYNTHETIC planted venue edge  : mean HAC t +3.84, fires |t|>=2 on 8/8 seeds


SYNTHETIC null (no edge)      : mean HAC t +0.55, fires |t|>=2 on 1/8 seeds


The detector fires on **every** planted seed and drops to roughly the nominal false-positive rate (1 seed in 8) once the edge is removed. The zero we measured on the real tape is a property of the tape, not a broken test.

## Verdict

- **Signal — None.** Over 538 real fills the open-versus-close slippage is **+1.39 bps per trade** (*t* = +0.33 naive, +0.23 clustered on the trade date) and the open fill wins **49.6%** of the time, 95% CI [45.4%, 53.8%] and [44.1%, 55.1%] clustered. The monthly and weekly rules print opposite signs on the same tapes; the era cut disagrees with itself; one cell in eight clears |*t*| = 2, which is what chance delivers.
- **Tradability — Mirage.** There is no sign to commit to in advance, so no venue rule can be written down. What survives is the noise: ±0.4 pp/yr of free track-record luck. And once the opening auction is charged its real, wider spread both rules drift toward the **close** — so fill at the close because it is cheaper and deeper, and expect exactly nothing for it.